**Identificações:**
* Enrico Minto Spanier - enrico.spanier@mackenzista.com.br - 10419775
* Guilherme Vecchi Bonotti Freitas Silveira - guilherme.silveir@mackenzista.com.br - 10418517
* Alexandre Luppi Severo e Silva - alexandreluppi.silva@mackenzista.com.br - 10419724

**Síntese do conteúdo:** Este notebook carrega a matriz de interações limpa, divide os dados em treino e teste, constrói um modelo de Deep Learning utilizando camadas de Embedding (TensorFlow/Keras) para usuários e itens, realiza o treinamento e avalia o erro do modelo utilizando a métrica RMSE.

In [ ]:
# Importação das bibliotecas para Deep Learning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate

### 1. Carga dos Dados e Separação (Train/Test)

In [ ]:
# Carregando o dataset preparado na etapa N1
df = pd.read_csv('dados/dataset_preparado.csv')

# Definindo as quantidades únicas para configurar as camadas de Embedding
num_users = df['user_idx'].nunique()
num_items = df['item_idx'].nunique()

print(f"Usuários: {num_users} | Produtos: {num_items}")

# Separação: 80% treino, 20% teste
X = df[['user_idx', 'item_idx']].values
y = df['review_score'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. Construção da Arquitetura Deep Learning
O modelo utilizará camadas de *Embedding* de 50 dimensões para capturar os padrões latentes tanto dos consumidores quanto dos produtos.

In [ ]:
# Dimensão do espaço vetorial (Embedding)
embedding_dim = 50

# Input para Usuários
user_input = Input(shape=(1,), name='user_input')
user_embedding = Embedding(input_dim=num_users, output_dim=embedding_dim, name='user_embedding')(user_input)
user_vec = Flatten(name='user_flatten')(user_embedding)

# Input para Produtos (Itens)
item_input = Input(shape=(1,), name='item_input')
item_embedding = Embedding(input_dim=num_items, output_dim=embedding_dim, name='item_embedding')(item_input)
item_vec = Flatten(name='item_flatten')(item_embedding)

# Concatenação e Camadas Densas Profundas
concat = Concatenate()([user_vec, item_vec])
dense_1 = Dense(128, activation='relu')(concat)
dense_2 = Dense(64, activation='relu')(dense_1)
output = Dense(1, activation='linear')(dense_2) # Previsão da nota (regressão)

# Compilando o Modelo
model = Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

### 3. Treinamento do Modelo
Vamos treinar o modelo observando a métrica de Perda Quadrática Média (MSE). *Nota: Por questões de tempo e poder computacional (GPU), o número de épocas foi reduzido.*

In [ ]:
# Treinamento
history = model.fit(
    x=[X_train[:, 0], X_train[:, 1]],
    y=y_train,
    batch_size=256,
    epochs=5,
    validation_split=0.1,
    verbose=1
)

### 4. Avaliação e Resultados (Métricas)
Calculamos o Erro Quadrático Médio da Raiz (RMSE) no conjunto de teste. A fórmula matemática da métrica é definida como $RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$.

In [ ]:
# Plot da curva de aprendizado (Loss)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Curva de Aprendizado - Loss (MSE)')
plt.ylabel('Erro')
plt.xlabel('Época')
plt.legend()
plt.show()

# Previsões no conjunto de teste
predictions = model.predict([X_test[:, 0], X_test[:, 1]])

# Cálculo do RMSE
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"RMSE no conjunto de Teste: {rmse:.4f}")

### 5. Simulação Prática de Recomendação
Selecionamos um usuário aleatório e prevemos quais produtos do catálogo ele provavelmente avaliaria com as maiores notas (as melhores recomendações).

In [ ]:
# Simulação para o Usuário de ID = 100
target_user = 100

# Pegamos um amostra de produtos que ele ainda não comprou (aqui simulado com os primeiros 50 produtos)
sample_products = np.arange(0, 50)
user_array = np.array([target_user] * len(sample_products))

# Prevendo as notas
predicted_ratings = model.predict([user_array, sample_products])

# Combinando e ordenando para pegar as top 3 recomendações
recomendacoes = pd.DataFrame({
    'Produto_Idx': sample_products,
    'Nota_Prevista': predicted_ratings.flatten()
})

top_3 = recomendacoes.sort_values(by='Nota_Prevista', ascending=False).head(3)

print("\n--- TOP 3 RECOMENDAÇÕES PARA O USUÁRIO 100 ---")
print(top_3)